
#Task2, News Modeling

#Nahid, 101518575

# News Modeling

Topic modeling involves **extracting features from document terms** and using
mathematical structures and frameworks like matrix factorization and SVD to generate **clusters or groups of terms** that are distinguishable from each other and these clusters of words form topics or concepts

Topic modeling is a method for **unsupervised classification** of documents, similar to clustering on numeric data

These concepts can be used to interpret the main **themes** of a corpus and also make **semantic connections among words that co-occur together** frequently in various documents

Topic modeling can help in the following areas:
- discovering the **hidden themes** in the collection
- **classifying** the documents into the discovered themes
- using the classification to **organize/summarize/search** the documents

Frameworks and algorithms to build topic models:
- Latent semantic indexing
- Latent Dirichlet allocation
- Non-negative matrix factorization

## Latent Dirichlet Allocation (LDA)
The latent Dirichlet allocation (LDA) technique is a **generative probabilistic model** where each **document is assumed to have a combination of topics** similar to a probabilistic latent semantic indexing model

In simple words, the idea behind LDA is that of two folds:
- each **document** can be described by a **distribution of topics**
- each **topic** can be described by a **distribution of words**

### LDA Algorithm

- 1. For each document, **randomly initialize each word to one of the K topics** (k is chosen beforehand)
- 2. For each document D, go through each word w and compute:
    - **P(T |D)** , which is a proportion of words in D assigned to topic T
    - **P(W |T )** , which is a proportion of assignments to topic T over all documents having the word W
- **Reassign word W with topic T** with probability P(T |D)´ P(W |T ) considering all other words and their topic assignments

![LDA](https://raw.githubusercontent.com/subashgandyer/datasets/main/images/LDA.png)

### Steps
- Install the necessary library
- Import the necessary libraries
- Download the dataset
- Load the dataset
- Pre-process the dataset
    - Stop words removal
    - Email removal
    - Non-alphabetic words removal
    - Tokenize
    - Lowercase
    - BiGrams & TriGrams
    - Lemmatization
- Create a dictionary for the document
- Filter low frequency words
- Create an Index to word dictionary
- Train the Topic Model
- Predict on the dataset
- Evaluate the Topic Model
    - Model Perplexity
    - Topic Coherence
- Visualize the topics

### Install the necessary library

In [ ]:
! pip install pyLDAvis gensim spacy
!pip install numpy pandas nltk gensim scikit-learn matplotlib seaborn #Removed the indentation from this line
!pip install gensim

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 49.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.7/26.7 MB 24.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.6/38.6 MB 7.6 MB/s eta 0:00:00
  Attempting uninstall: scipy
    Found existing installation: scipy 1.14.1
    Uninstalling scipy-1.14.1:
      Successfully uninstalled scipy-1.14.1


### Import the libraries

In [ ]:
import numpy as np
import pandas as pd
import nltk
import gensim
import gensim.corpora as corpora
from gensim.models import LdaModel
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
import matplotlib.pyplot as plt
import seaborn as sns
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import string

# Download required NLTK datasets (Run once)
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


True

### Download the dataset
Dataset: https://raw.githubusercontent.com/subashgandyer/datasets/main/newsgroups.json

#### 20-Newsgroups dataset
- 11K newsgroups posts
- 20 news topics

In [ ]:
import os
import json
import pandas as pd

# Download the dataset using wget
os.system("wget https://raw.githubusercontent.com/subashgandyer/datasets/main/newsgroups.json -O newsgroups.json")

# Load the dataset into a variable
with open("newsgroups.json", 'r') as f:
    data = json.load(f)

# Now you can work with the 'data' variable
print(type(data))  # Output: <class 'list'>
print(len(data))  # Output: 11314

# Convert the list 'data' into a pandas DataFrame and assign it to 'df'
df = pd.DataFrame(data) #This line creates a dataframe named df using the list data

# Save to a CSV file
df.to_csv("newsgroups.csv", index=False)

# Save to a JSON file
df.to_json("newsgroups.json", orient="records")

<class 'dict'>
3


### Load the dataset

In [ ]:
import pandas as pd

# Load the dataset into a pandas DataFrame
df = pd.read_json("newsgroups.json", orient="records")

# Print some info to verify it's loaded
print(df.head())  # Display the first few rows
print(df.info())   # Display information about the DataFrame

                                             content  target  \
0  From: lerxst@wam.umd.edu (where's my thing)\nS...       7   
1  From: guykuo@carson.u.washington.edu (Guy Kuo)...       4   
2  From: twillis@ec.ecn.purdue.edu (Thomas E Will...       4   
3  From: jgreen@amber (Joe Green)\nSubject: Re: W...       1   
4  From: jcm@head-cfa.harvard.edu (Jonathan McDow...      14   

            target_names  
0              rec.autos  
1  comp.sys.mac.hardware  
2  comp.sys.mac.hardware  
3          comp.graphics  
4              sci.space  
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11314 entries, 0 to 11313
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   content       11314 non-null  object
 1   target        11314 non-null  int64 
 2   target_names  11314 non-null  object
dtypes: int64(1), object(2)
memory usage: 265.3+ KB
None


### Preprocess the data

### Email Removal

In [ ]:
import re

def remove_email(text):
    """Removes email addresses from text using regular expressions."""
    return re.sub(r'\S*@\S*\s?', '', text)

# Apply the email removal function to the 'content' column
df['content'] = df['content'].apply(remove_email)

# Print the first few rows of the processed content to verify
print(df['content'].head())

0    From: (where's my thing)\nSubject: WHAT car is...
1    From: (Guy Kuo)\nSubject: SI Clock Poll - Fina...
2    From: (Thomas E Willis)\nSubject: PB questions...
3    From: (Joe Green)\nSubject: Re: Weitek P9000 ?...
4    From: (Jonathan McDowell)\nSubject: Re: Shuttl...
Name: content, dtype: object


### Newline Removal

In [ ]:
def remove_newline(text):
    """Removes newlines and extra whitespace from text using regular expressions."""
    # Replace newline characters with spaces
    text = re.sub(r'\n', ' ', text)
    # Replace multiple spaces with a single space
    text = re.sub(r'\s+', ' ', text)
    return text

# Apply the newline removal function to the 'content' column
df['content'] = df['content'].apply(remove_newline)

# Print the first few rows of the processed content to verify
print(df['content'].head())

0    From: (wheres my thing) Subject: WHAT car is t...
1    From: (Guy Kuo) Subject: SI Clock Poll - Final...
2    From: (Thomas E Willis) Subject: PB questions....
3    From: (Joe Green) Subject: Re: Weitek P9000 ? ...
4    From: (Jonathan McDowell) Subject: Re: Shuttle...
Name: content, dtype: object


### Single Quotes Removal

In [ ]:
def remove_single_quotes(text):
    """Removes single quotes from text using the replace method."""
    return text.replace("'", "")

# Apply the single quotes removal function to the 'content' column
df['content'] = df['content'].apply(remove_single_quotes)

# Print the first few rows of the processed content to verify
print(df['content'].head())

0    From: (wheres my thing) Subject: WHAT car is t...
1    From: (Guy Kuo) Subject: SI Clock Poll - Final...
2    From: (Thomas E Willis) Subject: PB questions....
3    From: (Joe Green) Subject: Re: Weitek P9000 ? ...
4    From: (Jonathan McDowell) Subject: Re: Shuttle...
Name: content, dtype: object


### Tokenize
- Create **sent_to_words()**
    - Use **gensim.utils.simple_preprocess**
    - Use **generator** instead of an usual function

In [ ]:
import gensim

def sent_to_words(sentences):
    """
    Tokenizes sentences into words using gensim.utils.simple_preprocess.
    Implemented as a generator.

    Args:
        sentences: A list of sentences.

    Yields:
        A list of words for each sentence.
    """

    # deacc=True removes punctuations
    for sentence in sentences:
        yield(gensim.utils.simple_preprocess(str(sentence), deacc=True))

### Stop words Removal
- Extend the stop words corpus with the following words
    - from
    - subject
    - re
    - edu
    - use

In [ ]:
import gensim

def sent_to_words(sentences):
    """
    Tokenizes sentences into words using gensim.utils.simple_preprocess.
    Implemented as a generator.

    Args:
        sentences: A list of sentences.

    Yields:
        A list of words for each sentence.
    """
    for sentence in sentences:
        # deacc=True removes punctuations
        yield(gensim.utils.simple_preprocess(str(sentence), deacc=True))

import nltk
from nltk.corpus import stopwords

def remove_stopwords(texts):
    """
    Removes stop words from a list of tokenized texts.

    Args:
        texts: A list of tokenized texts (list of lists of words).

    Returns:
        A list of texts with stop words removed.
    """
    stop_words = set(stopwords.words('english'))
    # Add custom stop words
    stop_words.update(['from', 'subject', 're', 'edu', 'use'])

    return [[word for word in text if word not in stop_words] for text in texts]

# Tokenize the 'content' column and create 'content_tokens' column
df['content_tokens'] = list(sent_to_words(df['content']))

# Now apply remove_stopwords
df['content_nostop'] = remove_stopwords(df['content_tokens'])

# Print to see the results
print(df['content_nostop'].head())

0    [wheres, thing, car, nntp, posting, host, rac,...
1    [guy, kuo, si, clock, poll, final, call, summa...
2    [thomas, willis, pb, questions, organization, ...
3    [joe, green, weitek, organization, harris, com...
4    [jonathan, mcdowell, shuttle, launch, question...
Name: content_nostop, dtype: object


#### remove_stopwords( )

In [ ]:
import gensim

def sent_to_words(sentences):
    """
    Tokenizes sentences into words using gensim.utils.simple_preprocess.
    Implemented as a generator.

    Args:
        sentences: A list of sentences.

    Yields:
        A list of words for each sentence.
    """
    for sentence in sentences:
        # deacc=True removes punctuations
        yield(gensim.utils.simple_preprocess(str(sentence), deacc=True))

import nltk
from nltk.corpus import stopwords

def remove_stopwords(texts):
    """
    Removes stop words from a list of tokenized texts.

    Args:
        texts: A list of tokenized texts (list of lists of words).

    Returns:
        A list of texts with stop words removed.
    """
    stop_words = set(stopwords.words('english'))
     # Add custom stop words
    stop_words.update(['from', 'subject', 're', 'edu', 'use'])

    return [[word for word in text if word not in stop_words] for text in texts]

# Tokenize the 'content' column and create 'content_tokens' column
df['content_tokens'] = list(sent_to_words(df['content']))

# Now apply remove_stopwords
df['content_nostop'] = remove_stopwords(df['content_tokens'])
print(df['content_nostop'])


0        [wheres, thing, car, nntp, posting, host, rac,...
1        [guy, kuo, si, clock, poll, final, call, summa...
2        [thomas, willis, pb, questions, organization, ...
3        [joe, green, weitek, organization, harris, com...
4        [jonathan, mcdowell, shuttle, launch, question...
                               ...                        
11309    [jim, zisfein, migraines, scans, distribution,...
11310    [screen, death, mac, plus, lines, organization...
11311    [estes, mounting, cpu, cooler, vertical, case,...
11312    [steven, collins, sphere, points, organization...
11313    [kevin, gunning, stolen, cbr, rr, organization...
Name: content_nostop, Length: 11314, dtype: object


### Bigrams
- Use **gensim.models.Phrases**
- 100 as threshold

In [ ]:
import gensim

def sent_to_words(sentences):
    for sentence in sentences:
        yield(gensim.utils.simple_preprocess(str(sentence), deacc=True))  # deacc=True removes punctuations

import nltk
from nltk.corpus import stopwords

def remove_stopwords(texts):
    stop_words = set(stopwords.words('english'))
    stop_words.update(['from', 'subject', 're', 'edu', 'use'])  # Add custom stop words
    return [[word for word in text if word not in stop_words] for text in texts]

df['content_tokens'] = list(sent_to_words(df['content']))
df['content_nostop'] = remove_stopwords(df['content_tokens'])

from gensim.models import Phrases

# Build the bigram models with a lower threshold
bigram = Phrases(df['content_nostop'], min_count=5, threshold=10)  # Lower threshold to detect more phrases

# Update the bigram model with the data
bigram.add_vocab(df['content_nostop'])  # This is important for detecting phrases correctly

# Faster way to get a sentence clubbed as a trigram/bigram
bigram_mod = gensim.models.phrases.Phraser(bigram)

def make_bigrams(texts):
    return [bigram_mod[doc] for doc in texts]

data_words_bigrams = make_bigrams(df['content_nostop'])

print(data_words_bigrams[0])

['wheres', 'thing', 'car', 'nntp_posting', 'host_rac', 'wam_umd', 'organization_university', 'maryland_college', 'park', 'lines', 'wondering_anyone', 'could_enlighten', 'car', 'saw', 'day', 'door_sports', 'car', 'looked', 'late_early', 'called', 'bricklin', 'doors', 'really', 'small', 'addition', 'front_bumper', 'separate', 'rest', 'body', 'know', 'anyone', 'tellme', 'model', 'name', 'engine', 'specs', 'years', 'production', 'car', 'made', 'history', 'whatever', 'info', 'funky_looking', 'car', 'please_mail', 'thanks', 'il', 'brought', 'neighborhood', 'lerxst']


#### make_bigrams( )

In [31]:
import gensim

# ... (other functions and code) ...

def make_bigrams(texts):
    return [bigram_mod[doc] for doc in texts]

# The following line was causing the error by overwriting make_bigrams function
#def make_bigrams(texts):
#    return Nonde

data_words_bigrams = make_bigrams(df['content_nostop'])

print(data_words_bigrams[0])

['wheres', 'thing', 'car', 'nntp_posting', 'host_rac', 'wam_umd', 'organization_university', 'maryland_college', 'park', 'lines', 'wondering_anyone', 'could_enlighten', 'car', 'saw', 'day', 'door_sports', 'car', 'looked', 'late_early', 'called', 'bricklin', 'doors', 'really', 'small', 'addition', 'front_bumper', 'separate', 'rest', 'body', 'know', 'anyone', 'tellme', 'model', 'name', 'engine', 'specs', 'years', 'production', 'car', 'made', 'history', 'whatever', 'info', 'funky_looking', 'car', 'please_mail', 'thanks', 'il', 'brought', 'neighborhood', 'lerxst']


### Lemmatization
- Use spacy
    - Download spacy en model (if you have not done that before)
    - Load the spacy model

In [32]:
import spacy

# Load the spaCy model using its full name
nlp = spacy.load("en_core_web_sm", disable=['parser', 'ner'])  # Correct way to load model


#### lemmatizaton( )

In [33]:
def lemmatization(texts, allowed_postags=['NOUN', 'ADJ', 'VERB', 'ADV']):
    """https://spacy.io/api/annotation"""
    texts_out = []
    for sent in texts:
        doc = nlp(" ".join(sent))
        texts_out.append([token.lemma_ for token in doc if token.pos_ in allowed_postags])
    return texts_out

In [34]:
data_lemmatized = lemmatization(data_words_bigrams, allowed_postags=['NOUN', 'ADJ', 'VERB', 'ADV'])

In [35]:
print(data_lemmatized[:1])

[['s', 'thing', 'car', 'nntp_poste', 'wam_umd', 'organization_university', 'maryland_college', 'park', 'line', 'wondering_anyone', 'car', 'see', 'day', 'door_sport', 'car', 'look', 'late_early', 'call', 'door', 'really', 'small', 'addition', 'separate', 'rest', 'body', 'know', 'model', 'name', 'engine', 'spec', 'year', 'production', 'car', 'make', 'history', 'info', 'funky_looke', 'car', 'please_mail', 'thank', 'bring', 'neighborhood', 'lerxst']]


### Create a Dictionary

In [36]:
from gensim.corpora import Dictionary

# Create a Dictionary
id2word = Dictionary(data_lemmatized)

# Print the Dictionary
print(id2word)

Dictionary<74489 unique tokens: ['addition', 'body', 'bring', 'call', 'car']...>


### Create Corpus

In [37]:
# Create Corpus
texts = data_lemmatized

# Term Document Frequency
corpus = [id2word.doc2bow(text) for text in texts]

# View
print(corpus[:1])

[[(0, 1), (1, 1), (2, 1), (3, 1), (4, 5), (5, 1), (6, 1), (7, 1), (8, 1), (9, 1), (10, 1), (11, 1), (12, 1), (13, 1), (14, 1), (15, 1), (16, 1), (17, 1), (18, 1), (19, 1), (20, 1), (21, 1), (22, 1), (23, 1), (24, 1), (25, 1), (26, 1), (27, 1), (28, 1), (29, 1), (30, 1), (31, 1), (32, 1), (33, 1), (34, 1), (35, 1), (36, 1), (37, 1), (38, 1)]]


### Filter low-frequency words

In [38]:
# Filter low-frequency words
id2word.filter_extremes(no_below=15, no_above=0.5, keep_n=100000)

# Update the corpus after filtering
corpus = [id2word.doc2bow(text) for text in texts]

# View the updated corpus
print(corpus[:1])

[[(0, 1), (1, 1), (2, 1), (3, 1), (4, 5), (5, 1), (6, 1), (7, 1), (8, 1), (9, 1), (10, 1), (11, 1), (12, 1), (13, 1), (14, 1), (15, 1), (16, 1), (17, 1), (18, 1), (19, 1), (20, 1), (21, 1), (22, 1), (23, 1), (24, 1), (25, 1), (26, 1), (27, 1), (28, 1), (29, 1), (30, 1), (31, 1), (32, 1)]]


### Create Index 2 word dictionary

In [39]:
# Create Index 2 word dictionary
index_to_word = {v: k for k, v in id2word.token2id.items()}

# Print the first few entries of the dictionary
print(index_to_word[0])  # Prints the word corresponding to ID 0
print(index_to_word[1])  # Prints the word corresponding to ID 1
# ...and so on

addition
body


### Build a News Topic Model

#### LdaModel
- **num_topics** : this is the number of topics you need to define beforehand
- **chunksize** : the number of documents to be used in each training chunk
- **alpha** : this is the hyperparameters that affect the sparsity of the topics
- **passess** : total number of training assess

In [40]:
from gensim.models import LdaModel

# Build LDA model
lda_model = LdaModel(
    corpus=corpus,
    id2word=id2word,
    num_topics=10,  # Number of topics
    chunksize=100,  # Number of documents to be used in each training chunk
    alpha='auto',  # Hyperparameter for topic sparsity (can also be a value like 0.1)
    passes=10     # Total number of training passes
)

In [41]:
for index, topic in lda_model.print_topics():
    print(f"Topic #{index}: {topic}")

Topic #0: 0.025*"use" + 0.020*"system" + 0.012*"window" + 0.012*"file" + 0.011*"work" + 0.011*"need" + 0.011*"run" + 0.011*"get" + 0.010*"problem" + 0.009*"bit"
Topic #1: 0.018*"issue" + 0.017*"book" + 0.015*"group" + 0.013*"order" + 0.012*"also" + 0.011*"ask" + 0.011*"call" + 0.010*"answer" + 0.010*"question" + 0.009*"discussion"
Topic #2: 0.788*"ax" + 0.053*"max" + 0.013*"pin" + 0.010*"_" + 0.008*"cable" + 0.004*"ad" + 0.004*"connector" + 0.004*"forsale" + 0.003*"organization_netcom" + 0.003*"r"
Topic #3: 0.024*"get" + 0.019*"go" + 0.017*"good" + 0.017*"think" + 0.016*"well" + 0.015*"make" + 0.015*"say" + 0.014*"time" + 0.013*"see" + 0.013*"know"
Topic #4: 0.033*"gun" + 0.021*"kill" + 0.015*"carry" + 0.012*"city" + 0.011*"people" + 0.011*"armenian" + 0.010*"police" + 0.010*"child" + 0.010*"mac" + 0.009*"weapon"
Topic #5: 0.026*"people" + 0.017*"say" + 0.015*"reason" + 0.015*"believe" + 0.011*"evidence" + 0.011*"state" + 0.011*"claim" + 0.009*"fact" + 0.009*"mean" + 0.008*"right"
Topi

### Print the Keyword in the 10 topics

## Evaluation of Topic Models
- Model Perplexity
- Topic Coherence

### Model Perplexity

Model perplexity is a measurement of **how well** a **probability distribution** or probability model **predicts a sample**

In [42]:
# Calculate and print perplexity
print('\nPerplexity: ', lda_model.log_perplexity(corpus))


Perplexity:  -7.073279742504485


### Topic Coherence
Topic Coherence measures score a single topic by measuring the **degree of semantic similarity** between **high scoring words** in the topic.

In [43]:
from gensim.models import CoherenceModel

# Compute Coherence Score
coherence_model_lda = CoherenceModel(model=lda_model, texts=data_lemmatized, dictionary=id2word, coherence='c_v')
coherence_lda = coherence_model_lda.get_coherence()
print('\nCoherence Score: ', coherence_lda)


Coherence Score:  0.5180835898064461


### Visualize the Topic Model
- Use **pyLDAvis**
    - designed to help users **interpret the topics** in a topic model that has been fit to a corpus of text data
    - extracts information from a fitted LDA topic model to inform an interactive web-based visualization

In [44]:
import pyLDAvis
import pyLDAvis.gensim_models as gensimvis
pyLDAvis.enable_notebook()

# Visualize the topics
vis = gensimvis.prepare(lda_model, corpus, id2word)
vis

PreparedData(topic_coordinates=              x         y  topics  cluster       Freq
topic                                                
3      0.206866 -0.142597       1        1  26.966992
5      0.173973 -0.182845       2        1  15.113040
0      0.091378  0.193211       3        1  13.464542
9      0.110395  0.077298       4        1   9.793140
1      0.118548  0.060767       5        1   7.734503
2     -0.345939 -0.149546       6        1   7.256437
7      0.021683  0.121074       7        1   6.172274
6     -0.125005  0.115660       8        1   5.492860
8     -0.182383  0.163474       9        1   4.226483
4     -0.069515 -0.256495      10        1   3.779730, topic_info=              Term          Freq         Total Category  logprob  loglift
2796            ax  52444.000000  52444.000000  Default  30.0000  30.0000
11            line   5844.000000   5844.000000  Default  29.0000  29.0000
192   organization   4539.000000   4539.000000  Default  28.0000  28.0000
1430           max   3560.000000   3560.000000  Default  27.0000  27.0000
123         people   5523.000000   5523.000000  Default  26.0000  26.0000
...            ...           ...           ...      ...      ...      ...
1619          area    295.915529    876.380407  Topic10  -4.7638   2.1898
450           live    265.740465   1130.921237  Topic10  -4.8714   1.8272
123         people    370.077651   5523.426491  Topic10  -4.5402   0.5725
62          report    244.394936    865.136294  Topic10  -4.9551   2.0114
1363         white    211.151423    406.872952  Topic10  -5.1013   2.6196

[524 rows x 6 columns], token_table=      Topic      Freq        Term
term                             
2795      6  0.998390           _
1002      2  0.998742      accept
2450      5  0.983676      accord
2450     10  0.013145      accord
1119      2  0.997569         act
...     ...       ...         ...
174       4  0.997419  world_nntp
156       1  0.996229       worth
32        1  0.456226        year
32        8  0.498024        year
32       10  0.045347        year

[797 rows x 3 columns], R=30, lambda_step=0.01, plot_opts={'xlab': 'PC1', 'ylab': 'PC2'}, topic_order=[4, 6, 1, 10, 2, 3, 8, 7, 9, 5])